# RT-IoT2022 - One-Class SVM (Support Vector Machine)

Now we are going to apply the One-Class SVM algorithm to our dataset. The main steps are:

- load the preprocessed datasets
- train a baseline One-Class SVM using default hyperparameters
- evaluate the model on the validation set
- study the effect of 'nu' and 'gamma'
- select the best configuration using validation
- evaluate the selected model on the untouched test set
- analyze the detection rate for the different attack types

The model is trained only on normal IoT traffic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import OneClassSVM

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score
)



In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_val = pd.read_csv("../data/processed/X_val.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

train_metadata = pd.read_csv("../data/processed/train_metadata.csv")
val_metadata = pd.read_csv("../data/processed/val_metadata.csv")
test_metadata = pd.read_csv("../data/processed/test_metadata.csv")

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

In [ ]:
y_val = (val_metadata["Label"] == "Anomaly").astype(int)
y_test = (test_metadata["Label"] == "Anomaly").astype(int)

print("Validation labels:")
print(y_val.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

In [ ]:
baseline_model = OneClassSVM()
baseline_model.fit(X_train)
baseline_predictions = baseline_model.predict(X_val)

baseline_predictions = (baseline_predictions == -1).astype(int)

In [ ]:
cm = confusion_matrix(y_val, baseline_predictions)
print(cm)

In [ ]:
tn, fp, fn, tp = cm.ravel()

print("True negatives:", tn)
print("False positivess:", fp)
print("False negatives:", fn)
print("True positives:", tp)


In [ ]:
accuracy = accuracy_score(y_val, baseline_predictions)
precision = precision_score(y_val, baseline_predictions)
recall = recall_score(y_val, baseline_predictions)
f1 = f1_score(y_val, baseline_predictions)
balanced_accuracy = balanced_accuracy_score(y_val, baseline_predictions)

fpr = fp / (fp + tn)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Balanced Accuracy:", balanced_accuracy)
print("False Positive Rate:", fpr)


In [ ]:
baseline_scores = -baseline_model.decision_function(X_val)

roc_auc = roc_auc_score(y_val, baseline_scores)
pr_auc = average_precision_score(y_val, baseline_scores)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

## Hyperparameter analysis - parameter 'nu'

The 'nu' parameter is very important because it influences the One-Class SVM boundary decision. A larger nu generally allows more training observations to fall outside the learned normal region and viceversa for a smaller nu.

The default value is '0.5', but this value SHOULD NOT be accepted without analysis because the training dataset should contain ONLY normal traffic.

## Hyperparameter analysis - parameter 'gamma'

The RBF kernel is used to learn a non-linear boundary around normal traffic. 

The goal of the 'gamma' parameter is to control how strongly individual training samples influence the shape of this boundary.

So, small gamma values tend to produce smoother boundaries while larger values can create more complex and local boundaries.

In this project, both nu and gamma are analyzed instead of simply using their default values.

In [ ]:
nu_values = [0.01, 0.05, 0.10, 0.20, 0.50]
gamma_values = ["scale", 0.001, 0.01, 0.1]

In [ ]:
results = []

for nu in nu_values:
    for gamma in gamma_values:
        
        print("Testing nu =", nu, "gamma =", gamma)
        
        model = OneClassSVM(
            kernel = "rbf",
            nu = nu,
            gamma = gamma  
        )
        
        model.fit(X_train)
        
        predictions = model.predict(X_val)
        predictions = (predictions == -1).astype(int)
        
        cm = confusion_matrix(y_val, predictions)
        tn, fp, fn, tp = cm.ravel()
        
        precision = precision_score(y_val, predictions)
        recall = recall_score(y_val, predictions)
        f1 = f1_score(y_val, predictions)
        balanced_accuracy = balanced_accuracy_score(
            y_val,
            predictions
        )
        
        fpr = fp / (fp + tn)
        
        scores = -model.decision_function(X_val)
        
        roc_auc = roc_auc_score(y_val, scores)
        pr_auc = average_precision_score(y_val, scores)
        
        results.append({
            "nu": nu,
            "gamma": gamma,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "balanced_accuracy": balanced_accuracy,
            "fpr": fpr,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc
        })

In [ ]:
results_df = pd.DataFrame(results)

results_df

results_df = results_df.sort_values(
    by="balanced_accuracy",
    ascending=False
)

results_df

In [ ]:
results_df.head(10)
best_result = results_df.iloc[0]

print("Best nu:", best_result["nu"])
print("Best gamma:", best_result["gamma"])
print("Balanced accuracy:", best_result["balanced_accuracy"])
print("F1-score:", best_result["f1"])
print("Recall:", best_result["recall"])
print("False Positive Rate:", best_result["fpr"])

In [ ]:
scale_results = results_df[
    results_df["gamma"] == "scale"
].sort_values("nu")

plt.figure(figsize=(7,5))
plt.plot(
    scale_results["nu"],
    scale_results["balanced_accuracy"],
    marker="o"
)

plt.xlabel("nu")
plt.ylabel("Balanced Accuracy")
plt.title("Effect of nu on One-Class SVM")
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(7,5))

plt.scatter(
    results_df["fpr"],
    results_df["recall"]
)

plt.xlabel("False Positive Rate")
plt.ylabel("Recall")
plt.title("One-Class SVM - Recall vs False Positive Rate")
plt.grid()
plt.show()

In [ ]:
best_nu = results_df.iloc[0]["nu"]
best_gamma = results_df.iloc[0]["gamma"]

print("Selected nu:", best_nu)
print("Selected gamma:", best_gamma)

In [ ]:
final_model = OneClassSVM(
    kernel="rbf",
    nu = best_nu,
    gamma = best_gamma
)

final_model.fit(X_train)

In [ ]:
test_predictions = final_model.predict(X_test)
test_predictions = (test_predictions == -1).astype(int)

test_cm = confusion_matrix(y_test, test_predictions)

tn,fp,fn,tp = test_cm.ravel()

print("Confusion Matrix:")
print(test_cm)

print("\nTrue Negaitves:", tn)
print("False Positives:", fp)
print("False Negaitves:", fn)
print("True Positives:", tp)

In [ ]:
test_accuracy = accuracy_score(y_test, test_predictions)
test_precision = precision_score(y_test, test_predictions)
test_recall = recall_score(y_test, test_predictions)
test_f1 = f1_score(y_test, test_predictions)
test_balanced_accuracy = balanced_accuracy_score(y_test, test_predictions)

test_fpr = fp / (fp + tn)

test_scores = -final_model.decision_function(X_test)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)

In [ ]:
print("Accuracy:", test_accuracy)
print("Precision:", test_precision)
print("Recall:", test_recall)
print("F1-score:", test_f1)
print("Balanced Accuracy:", test_balanced_accuracy)
print("False Positive Rate:", test_fpr)
print("ROC-AUC:", test_roc_auc)
print("PR-AUC:", test_pr_auc)

In [ ]:
attack_analysis = test_metadata.copy()
attack_analysis["Prediction"] = test_predictions

attack_analysis = attack_analysis[
    attack_analysis["Label"] == "Anomaly"
]

attack_detection = attack_analysis.groupby(
    "Attack_type"
)["Prediction"].agg(["count", "sum"])

attack_detection["detection_rate"] = (
    attack_detection["sum"] /
    attack_detection["count"]
)

attack_detection

In [ ]:
normal_analysis = test_metadata.copy()
normal_analysis["Prediction"] = test_predictions
normal_analysis = normal_analysis[normal_analysis["Label"] == "Normal"]

false_alarms = normal_analysis.groupby("Attack_type")["Prediction"].agg(["count", "sum"])
false_alarms.columns = ["normal_samples", "false_alarms"] 

false_alarms["false_positive_rate"] = (
    false_alarms["false_alarms"] / false_alarms["normal_samples"]
)

false_alarms

In [ ]:
results_df.to_csv(
    "../results/tables/one_class_svm_validation_results.csv",
    index=False
)

attack_detection.to_csv(
    "../results/tables/one_class_svm_attack_detection.csv"
)

In [ ]:
tn, fp, fn, tp = test_cm.ravel()

final_metrics = pd.DataFrame([{
    "model": "One-Class SVM",
    "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    "accuracy": test_accuracy,
    "precision": test_precision,
    "recall": test_recall,
    "f1": test_f1,
    "balanced_accuracy": test_balanced_accuracy,
    "fpr": test_fpr,
    "roc_auc": test_roc_auc,
    "pr_auc": test_pr_auc
}])

final_metrics.to_csv("../results/tables/one_class_svm_final_metrics.csv", index=False)
false_alarms.to_csv("../results/tables/one_class_svm_false_alarms.csv")
final_metrics

In [ ]:
port_cols = ["id.orig_p", "id.resp_p"]

svm_no_ports = OneClassSVM(kernel = "rbf", nu = best_nu, gamma = best_gamma)
svm_no_ports.fit(X_train.drop(columns = port_cols))

no_ports_predictions = (svm_no_ports.predict(X_test.drop(columns = port_cols)) == -1).astype(int)

print("Balanced Accuracy senza porte:", balanced_accuracy_score(y_test, no_ports_predictions))
print("Recall senza porte:", recall_score(y_test, no_ports_predictions))